In [1]:
import pandas as pd
import numpy as np

train_file = open('./LightGCN/train.txt', 'r')
test_file = open('./LightGCN/test.txt', 'r')

uid_ls, his_ls = [], []

for train_line, test_line in zip(train_file.readlines(), test_file.readlines()):
    tmp_train_line = train_line.strip().split(' ')
    tmp_test_line = test_line.strip().split(' ')

#     uid = int(tmp_train_line[0])
    uid = int(tmp_train_line[0]) + 1 #因为是从lightGCN从0开始编码，但是我们要把0当作padding

    assert tmp_train_line[0] == tmp_test_line[0]

    his = [int(iid)+1 for iid in tmp_train_line[1:]] + \
            [int(iid)+1 for iid in tmp_test_line[1:]]
    
    uid_ls.append(uid)
    his_ls.append(his)




In [2]:
use_all_his = pd.DataFrame(
    data = zip(uid_ls, his_ls), columns=['user_id', 'user_history']
)

use_all_his

,user_id,user_history
0,1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,2,"[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3..."
2,3,"[554, 555, 556, 557, 558, 559, 560, 561, 562, ..."
3,4,"[561, 559, 600, 555, 566, 622, 573, 578, 588, ..."
4,5,"[625, 626, 627, 628, 629, 630, 631, 632, 633, ..."
...,...,...
31663,31664,"[32941, 36662, 28423, 11153, 37681, 3207, 3536..."
31664,31665,"[11125, 7366, 16965, 21421, 16885, 24961, 1789..."
31665,31666,"[8846, 9847, 10252, 26225, 14973, 11528, 13748..."
31666,31667,"[37463, 29817, 2356, 23560, 9633, 13381, 6401,..."


In [3]:
import ast
df = use_all_his.copy()

# Flatten all lists in 'user_history' column
all_ids = [item for sublist in df['user_history'] for item in sublist]

# Get unique values and count
unique_ids = set(all_ids)
unique_ids_count = len(unique_ids)

print("Total number of unique IDs:", unique_ids_count)


Total number of unique IDs: 38048


In [4]:
num_users = df['user_id'].nunique()
print("Number of unique users:", num_users)

Number of unique users: 31668


In [6]:
use_all_his.to_csv('./dataset/user_with_full_his.tsv', sep='\t', index=False)

In [7]:
import numpy as np

all_user_item_inter_pd = use_all_his.copy()

all_users = all_user_item_inter_pd.shape[0]
test_val_users = np.random.choice(all_users, int(all_users * 0.2), replace=False)
test = test_val_users[:len(test_val_users)//2]
val = test_val_users[len(test_val_users)//2:]

training_data = all_user_item_inter_pd[
    ~all_user_item_inter_pd['user_id'].isin(test_val_users)
].copy()
val_data = all_user_item_inter_pd[
    all_user_item_inter_pd['user_id'].isin(val)
].copy()
test_data = all_user_item_inter_pd[
    all_user_item_inter_pd['user_id'].isin(test)
].copy()

training_data.shape, val_data.shape, test_data.shape

((25335, 2), (3167, 2), (3166, 2))

In [8]:
val_data.head()

,user_id,user_history
26,27,"[1315, 1055, 1316, 1061, 1317, 1255, 1318, 131..."
32,33,"[1665, 836, 1666, 1667, 1668, 559, 1669, 609, ..."
33,34,"[1529, 817, 663, 784, 1683, 650, 88, 689, 1530..."
46,47,"[683, 808, 1541, 1543, 847, 849, 12, 858, 860,..."
52,53,"[2947, 666, 1541, 2948, 2949, 2950, 2951, 2952..."


In [9]:
val_data['given_user_history'] = val_data['user_history'].apply(
    lambda x:
        x[:int(0.8*len(x))] 
        # if len(x) >= 10
        # else x[:-2]
)
val_data['predicting_items'] = val_data['user_history'].apply(
    lambda x:
        x[int(0.8*len(x)):] 
        # if len(x) >= 10
        # else x[-2:]
)

In [10]:
test_data['given_user_history'] = test_data['user_history'].apply(
    lambda x:
        x[:int(0.8*len(x))] 
        # if len(x) >= 10
        # else x[:-2]
)
test_data['predicting_items'] = test_data['user_history'].apply(
    lambda x:
        x[int(0.8*len(x)):] 
        # if len(x) >= 10
        # else x[-2:]
)

In [11]:
val_data.head()

,user_id,user_history,given_user_history,predicting_items
26,27,"[1315, 1055, 1316, 1061, 1317, 1255, 1318, 131...","[1315, 1055, 1316, 1061, 1317, 1255, 1318, 131...","[11919, 4631, 1640, 1639, 8275, 8903, 7763]"
32,33,"[1665, 836, 1666, 1667, 1668, 559, 1669, 609, ...","[1665, 836, 1666, 1667, 1668, 559, 1669, 609, ...","[990, 4694, 991, 623, 7662, 4675, 1313, 574, 1..."
33,34,"[1529, 817, 663, 784, 1683, 650, 88, 689, 1530...","[1529, 817, 663, 784, 1683, 650, 88, 689, 1530...","[2741, 772, 773, 2867]"
46,47,"[683, 808, 1541, 1543, 847, 849, 12, 858, 860,...","[683, 808, 1541, 1543, 847, 849, 12, 858, 860,...","[20269, 91, 28984, 3649, 2726, 7964, 10385, 12..."
52,53,"[2947, 666, 1541, 2948, 2949, 2950, 2951, 2952...","[2947, 666, 1541, 2948, 2949, 2950, 2951, 2952...","[9491, 8529, 3333, 5159, 21717, 472, 4511, 10590]"


In [12]:
val_data.drop(columns=['user_history']).to_csv(
    './dataset/validation.tsv', sep='\t', index=False
)
test_data.drop(columns=['user_history']).to_csv(
    './dataset/test.tsv', sep='\t', index=False
)

In [13]:
training_data.to_csv(
    './dataset/training_with_full_his.tsv', sep='\t', index=False
)
training_data['user_history'].loc[0]

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 796,
 695,
 1532,
 14518,
 8785]

In [19]:
training_data.head(10)

,user_id,user_history
0,1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,2,"[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3..."
3,4,"[561, 559, 600, 555, 566, 622, 573, 578, 588, ..."
6,7,"[663, 664, 665, 666, 667, 668, 669, 670, 671, ..."
7,8,"[745, 746, 637, 747, 748, 749, 750, 751, 752, ..."
8,9,"[626, 634, 638, 644, 760, 689, 761, 762, 763, ..."
9,10,"[779, 780, 781, 782, 783, 784, 16, 785, 786, 7..."
11,12,"[814, 815, 816, 817, 818, 819, 687, 83, 820, 7..."
12,13,"[610, 568, 574, 829, 622, 597, 586, 624, 594, ..."
13,14,"[843, 844, 845, 654, 846, 698, 697, 847, 787, ..."


In [ ]:
import pandas as pd

training_his = training_data
# training_his = pd.read_csv(
#     './dataset/training.tsv', sep='\t'
# )

uid_ls, user_his_ls, next_item_ls = [], [], []

for _, user_table in training_his.iterrows():
    cur_reco_his = user_table['user_history']
    # cur_reco_his = eval(user_table['user_history'])
    for i in range(1, len(cur_reco_his)):
        user_his_ls.append(cur_reco_his[:i])
        uid_ls.append(int(user_table['user_id']))
        next_item_ls.append(cur_reco_his[i])

training_pd = pd.DataFrame(
    data=zip(uid_ls, user_his_ls, next_item_ls), columns=['user_id', 'user_history', 'next_item']
)